## 1. Imports

In [ ]:
import math
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
from tqdm import tqdm
from typing import Dict, List, Optional, Tuple
from collections import OrderedDict

from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import torch.optim as optim

import flwr as fl

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 2. Hyperparameters

**Fix #2**: `INPUT_SIZE` now matches the number of columns in `cols` (5).  
**Fix #6**: `NUM_EPOCHS` (local epochs per round) reduced from 100-250 to 5 to prevent client drift.

In [ ]:
cols = ['throughput', 'sinr', 'dlpathloss', 'delay', 'pdr']

target_feature_names = ['throughput', 'delay']

# --- Architecture ---
INPUT_SIZE = len(cols)         
OUTPUT_SIZE = len(target_feature_names)  # number of target features
HIDDEN_SIZE = 256
NUM_LAYERS = 3

# --- Training ---
BATCH_SIZE = 64
LEARNING_RATE = 0.001
NUM_EPOCHS = 5

# --- Windowing ---
FUTURE_WIN_LEN = 5
WINDOW_LEN = 15
STRIDE = FUTURE_WIN_LEN         # non-overlapping targets
FILTER_LEN = 5

# --- Federated ---
NUM_CLIENTS = 7
NUM_ROUNDS = 100

RESULTS_DIR = "../results/"
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"INPUT_SIZE={INPUT_SIZE}, OUTPUT_SIZE={OUTPUT_SIZE}")
print(f"Local epochs per round: {NUM_EPOCHS}")
print(f"Window: {WINDOW_LEN} -> predict next {FUTURE_WIN_LEN} steps")

## 3. Data Loading & Imputation

In [ ]:
df = pd.read_csv('./../ML-Aided-NR-V2X-QoS-Predictions/dataset.csv')
print(f"Original shape: {df.shape}")

df.drop(columns='Unnamed: 0', inplace=True)
df.drop_duplicates(inplace=True)
print(f"After dedup: {df.shape}")

print(f"\nMissing values:")
print(df[['distance', 'jitter']].isnull().sum())

In [ ]:
# Impute missing 'distance' using linear regression on 'pdr'
clean = df[['distance', 'pdr']].dropna()
reg = LinearRegression()
reg.fit(clean[['pdr']], clean[['distance']])
missing_dist = df['distance'].isnull()
df.loc[missing_dist, 'distance'] = reg.predict(df.loc[missing_dist, ['pdr']])

# Impute missing 'jitter' using linear regression on 'delay'
clean = df[['jitter', 'delay']].dropna()
reg = LinearRegression()
reg.fit(clean[['delay']], clean[['jitter']])
missing_jitter = df['jitter'].isnull()
df.loc[missing_jitter, 'jitter'] = reg.predict(df.loc[missing_jitter, ['delay']])

print(f"Missing values after imputation: {df[cols].isnull().sum().sum()}")

## 4. Smoothing (Non-Target Features Only)

In [ ]:
cols_to_smoothen = ['sinr', 'dlpathloss']

for col in cols_to_smoothen:
    df[col] = df[col].rolling(window=FILTER_LEN).mean()

df.bfill(inplace=True)
df.ffill(inplace=True)

print(f"Smoothed columns: {cols_to_smoothen}")
print(f"NOT smoothed (targets): {target_feature_names}")

## 5. Sliding Window & Dataset Creation

Targets are taken from **future** timesteps beyond the input window (no information leakage).

In [ ]:
def sliding_window_augmentation(df, cols, indices, future_window_len=1, L=100, s=10, add_noise=True, noise_level=0.02):
    X = df[cols].values
    T, D = X.shape

    n_windows = max(0, (T - L - future_window_len) // s + 1)

    if n_windows <= 0:
        return np.array([]), np.array([])

    num_targets = len(indices)
    windows = np.zeros((n_windows, L, D))
    targets = np.zeros((n_windows, future_window_len, num_targets))

    col_stds = df[cols].std().values

    for i in range(n_windows):
        start = i * s
        end_input = start + L
        end_target = end_input + future_window_len

        w = X[start:end_input].copy()
        target = X[end_input:end_target, indices].copy()

        if add_noise:
            input_noise = np.random.randn(*w.shape) * col_stds * noise_level
            w += input_noise
            target_noise = np.random.randn(future_window_len, num_targets) * col_stds[indices] * noise_level
            target += target_noise

        windows[i] = w
        targets[i] = target

    return windows, targets

In [ ]:
class QosDataset(Dataset):
    def __init__(self, windowed_data, target):
        self.X = torch.tensor(windowed_data, dtype=torch.float32)
        self.y = torch.tensor(target, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
def create_autoregressive_dataset(data, cols, indices, future_window_len=1, window_len=100, stride=10):
    """Create clean + noisy augmented windows."""
    windows, targets = sliding_window_augmentation(
        data, cols, indices, future_window_len=future_window_len, L=window_len, s=stride, add_noise=False
    )

    window_noisy, target_noisy = sliding_window_augmentation(
        data, cols, indices, future_window_len=future_window_len, L=window_len, s=stride, add_noise=True
    )

    if len(windows) > 0 and len(window_noisy) > 0:
        all_windows = np.concatenate([windows, window_noisy], axis=0)
        all_targets = np.concatenate([targets, target_noisy], axis=0)
    elif len(windows) > 0:
        all_windows = windows
        all_targets = targets
    else:
        raise ValueError("No valid windows created")

    print(f"Created {len(all_windows)} windows (clean: {len(windows)}, noisy: {len(window_noisy)})")
    print(f"Window shape: {all_windows.shape}, Target shape: {all_targets.shape}")

    return all_windows, all_targets

In [ ]:
# Build target indices from cols
indices = [cols.index(name) for name in target_feature_names]
print(f"Target features: {target_feature_names} at indices {indices}")

all_windows, all_targets = create_autoregressive_dataset(
    df, cols, indices, future_window_len=FUTURE_WIN_LEN, window_len=WINDOW_LEN, stride=STRIDE
)

## 6. Federated Data Partitioning

In [ ]:
def create_federated_data(all_windows, all_targets, num_clients, batch_size, train_test_split_ratio=0.8):
    """Split data across clients preserving temporal order.
    
    FIX #1: No shuffling before client partitioning.
    FIX #8: Returns a global test set from all clients.
    """
    client_loaders = []
    global_test_X_parts = []
    global_test_y_parts = []
    
    num_samples = len(all_windows)
    samples_per_client = num_samples // num_clients

    for i in range(num_clients):
        start_idx = i * samples_per_client
        end_idx = (i + 1) * samples_per_client if i < num_clients - 1 else num_samples

        client_windows = all_windows[start_idx:end_idx]
        client_targets = all_targets[start_idx:end_idx]

        if len(client_windows) < 2:
            continue

        # Chronological split — no shuffle for time series
        X_train, X_test, y_train, y_test = train_test_split(
            client_windows, client_targets, train_size=train_test_split_ratio, shuffle=False
        )

        # Scale input features
        scaler = MinMaxScaler()
        n_train, n_timesteps, n_features = X_train.shape
        X_train_scaled = scaler.fit_transform(X_train.reshape(-1, n_features)).reshape(n_train, n_timesteps, n_features)
        
        n_test = X_test.shape[0]
        X_test_scaled = scaler.transform(X_test.reshape(-1, n_features)).reshape(n_test, n_timesteps, n_features)

        # Scale targets
        target_scaler = MinMaxScaler()
        n_train_y, n_timesteps_y, n_features_y = y_train.shape
        y_train_scaled = target_scaler.fit_transform(y_train.reshape(-1, n_features_y)).reshape(n_train_y, n_timesteps_y, n_features_y)
        
        n_test_y = y_test.shape[0]
        y_test_scaled = target_scaler.transform(y_test.reshape(-1, n_features_y)).reshape(n_test_y, n_timesteps_y, n_features_y)

        train_dataset = QosDataset(X_train_scaled, y_train_scaled)
        test_dataset = QosDataset(X_test_scaled, y_test_scaled)

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

        client_loaders.append({
            'train': train_loader,
            'test': test_loader,
            'target_scaler': target_scaler,
            'feature_scaler': scaler
        })

        # FIX #8: Collect test data for global evaluation
        global_test_X_parts.append(X_test_scaled)
        global_test_y_parts.append(y_test_scaled)

        print(f"Client {i}: train={len(train_dataset)}, test={len(test_dataset)}")

    # Create combined global test set
    global_test_X = np.concatenate(global_test_X_parts, axis=0)
    global_test_y = np.concatenate(global_test_y_parts, axis=0)
    global_test_dataset = QosDataset(global_test_X, global_test_y)
    global_test_loader = DataLoader(global_test_dataset, batch_size=batch_size, shuffle=False)

    print(f"\nGlobal test set: {len(global_test_dataset)} samples")

    return client_loaders, global_test_loader

In [ ]:
client_loaders, global_test_loader = create_federated_data(
    all_windows=all_windows,
    all_targets=all_targets,
    num_clients=NUM_CLIENTS,
    batch_size=BATCH_SIZE
)

global_target_scaler = client_loaders[0]["target_scaler"]

## 7. Model Utilities

In [ ]:
def get_parameters(model) -> List[np.ndarray]:
    return [val.cpu().numpy() for _, val in model.state_dict().items()]

def set_parameters(model, parameters: List[np.ndarray]):
    param_dict = zip(model.state_dict().keys(), parameters)
    state_dict = OrderedDict({k: torch.from_numpy(v) for k, v in param_dict})
    model.load_state_dict(state_dict, strict=True)

## 8. LSTM Model

In [ ]:
class LSTM(nn.Module):
    def __init__(self, input_size=12, num_target_features=2, future_window_len=10,
                 hidden_size=50, num_layers=3, dropout=0.3, bidirectional=False):
        super(LSTM, self).__init__()
        self.num_target_features = num_target_features
        self.future_window_len = future_window_len
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = bidirectional

        self.lstm = nn.LSTM(
            input_size,
            hidden_size,
            num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional
        )

        direction_factor = 2 if bidirectional else 1

        self.fc = nn.Sequential(
            nn.Linear(hidden_size * direction_factor, 128),
            nn.ReLU(),
            nn.Linear(128, future_window_len * num_target_features)
        )

    def forward(self, x):
        h0 = torch.zeros(self.num_layers * (2 if self.bidirectional else 1), x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers * (2 if self.bidirectional else 1), x.size(0), self.hidden_size).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = out[:, -1, :]  # last time step
        out = self.fc(out)
        # Reshape to (batch, future_window_len, num_target_features)
        out = out.view(x.size(0), self.future_window_len, self.num_target_features)
        return out

## 9. Training & Validation Functions

In [ ]:
def train(model, trainloader, criterion, optimizer, epochs, scheduler=None):
    model.to(DEVICE)
    model.train()

    for epoch in range(epochs):
        epoch_loss = 0
        for X, y in trainloader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            pred = model(X)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        if scheduler is not None:
            scheduler.step()

    avg_loss = epoch_loss / len(trainloader)
    return avg_loss

In [ ]:
def validate(model, valloader, criterion, target_scaler):
    model.to(DEVICE)
    model.eval()
    epoch_loss = 0
    all_pred_unscaled = []
    all_targets_unscaled = []

    with torch.no_grad():
        for X, y in valloader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            pred = model(X)

            loss = criterion(pred, y)
            epoch_loss += loss.item()

            pred_cpu = pred.cpu().numpy()
            y_cpu = y.cpu().numpy()

            # FIX #5: num_target_features is the LAST dimension, not shape[1]
            # pred_cpu shape: (batch, future_window_len, num_target_features)
            num_target_features = pred_cpu.shape[-1]

            # Reshape to 2D for inverse_transform: (-1, num_target_features)
            pred_reshaped = pred_cpu.reshape(-1, num_target_features)
            y_reshaped = y_cpu.reshape(-1, num_target_features)

            # FIX #4: Always inverse-scale
            pred_unscaled = target_scaler.inverse_transform(pred_reshaped)
            y_unscaled = target_scaler.inverse_transform(y_reshaped)

            all_pred_unscaled.append(pred_unscaled)
            all_targets_unscaled.append(y_unscaled)

    avg_loss = epoch_loss / len(valloader)

    final_preds = np.concatenate(all_pred_unscaled)    # shape: (total_samples, num_target_features)
    final_targets = np.concatenate(all_targets_unscaled)

    # FIX #5: Compute R² per TARGET FEATURE (columns), not per time step
    num_features = final_targets.shape[1]  # This is now num_target_features after reshape
    r2_scores = {}
    for i in range(num_features):
        r2 = r2_score(final_targets[:, i], final_preds[:, i])
        r2_scores[f'r2_feature_{i}'] = r2

    return avg_loss, r2_scores

## 10. Flower Client

In [ ]:
class QosClient(fl.client.NumPyClient):
    """Flower client for QoS prediction."""

    def __init__(self, cid: str, model: nn.Module, trainloader, testloader,
                 target_scaler: MinMaxScaler, target_feature_names: list):
        self.cid = cid
        self.model = model.to(DEVICE)
        self.trainloader = trainloader
        self.testloader = testloader
        self.target_scaler = target_scaler
        self.target_feature_names = target_feature_names

        self.criterion = nn.MSELoss()
        self.optimizer = optim.Adam(self.model.parameters(), lr=LEARNING_RATE)
        self.last_train_loss: float = -1.0

    def get_parameters(self, config: dict) -> list:
        return get_parameters(self.model)

    def fit(self, parameters: list, config: dict) -> tuple:
        set_parameters(self.model, parameters)
        avg_train_loss = train(
            self.model, self.trainloader, self.criterion, self.optimizer, config["local_epochs"]
        )
        self.last_train_loss = avg_train_loss
        return get_parameters(self.model), len(self.trainloader.dataset), {"train_loss": avg_train_loss}

    def evaluate(self, parameters: list, config: dict) -> tuple:
        set_parameters(self.model, parameters)
        val_loss, r2_scores = validate(
            self.model, self.testloader, self.criterion, self.target_scaler
        )
        return float(val_loss), len(self.testloader.dataset), r2_scores

## 11. Federated Learning Setup

In [ ]:
def client_fn(cid) -> QosClient:
    model = LSTM(
        input_size=INPUT_SIZE,
        num_target_features=OUTPUT_SIZE,
        future_window_len=FUTURE_WIN_LEN,
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS
    ).to(DEVICE)
    trainloader = client_loaders[int(cid)]["train"]
    testloader = client_loaders[int(cid)]["test"]
    scaler = client_loaders[int(cid)]["target_scaler"]
    return QosClient(cid, model, trainloader, testloader, scaler, target_feature_names)

In [ ]:
# Initialize model to get initial parameters
model = LSTM(
    input_size=INPUT_SIZE,
    num_target_features=OUTPUT_SIZE,
    future_window_len=FUTURE_WIN_LEN,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS
).to(DEVICE)
params = get_parameters(model)
del model
print("Initial parameters extracted.")

In [ ]:
def evaluate_server(server_round: int, params: fl.common.NDArrays,
                    config: Dict[str, fl.common.Scalar]) -> Optional[Tuple[float, Dict[str, fl.common.Scalar]]]:
    """Server-side evaluation using global test set (FIX #8)."""
    model = LSTM(
        input_size=INPUT_SIZE,
        num_target_features=OUTPUT_SIZE,
        future_window_len=FUTURE_WIN_LEN,
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS
    ).to(DEVICE)

    set_parameters(model, params)
    criterion = torch.nn.MSELoss()

    loss, r2_scores = validate(model, global_test_loader, criterion, global_target_scaler)

    feature_map = {f"r2_feature_{i}": name for i, name in enumerate(target_feature_names)}

    print(f"Round {server_round}: loss={loss:.4f}", end="")
    for key, score in r2_scores.items():
        print(f" | R²({feature_map.get(key, key)})={score:.4f}", end="")
    print()

    return loss, r2_scores

In [ ]:
def fit_config(server_round: int):
    return {
        "server_round": server_round,
        "local_epochs": NUM_EPOCHS
    }

In [ ]:
strategy = fl.server.strategy.FedAvg(
    fraction_fit=0.7,
    fraction_evaluate=1.0,
    min_fit_clients=5,
    min_evaluate_clients=NUM_CLIENTS,
    min_available_clients=NUM_CLIENTS,
    initial_parameters=fl.common.ndarrays_to_parameters(params),
    evaluate_fn=evaluate_server,
    on_fit_config_fn=fit_config,
    on_evaluate_config_fn=fit_config
)

## 12. Run Simulation

In [ ]:
client_resources = {"num_cpus": 1, "num_gpus": 1}

Output_History = fl.simulation.start_simulation(
    client_fn=client_fn,
    num_clients=NUM_CLIENTS,
    config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
    strategy=strategy,
    client_resources=client_resources
)

## 13. Results & Visualization

In [ ]:
# Plot aggregated loss
rounds = [rnd for rnd, _ in Output_History.losses_distributed]
losses = [loss for _, loss in Output_History.losses_distributed]

plt.figure(figsize=(14, 5))
plt.plot(rounds, losses, marker='.', linestyle='-', markersize=3)
plt.title("Server-Side Aggregated Loss vs. Round", fontsize=16)
plt.xlabel("Federated Round", fontsize=12)
plt.ylabel("Aggregated Loss", fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "server_loss_plot.png"), dpi=150)
plt.show()

In [ ]:
# Extract and display best R² scores
for i, name in enumerate(target_feature_names):
    key = f'r2_feature_{i}'
    r2_history = Output_History.metrics_centralized[key]
    scores = [score for _, score in r2_history]
    rounds_r2 = [rnd for rnd, _ in r2_history]
    
    best_idx = np.argmax(scores)
    print(f"Best R² for {name}: {scores[best_idx]:.4f} (round {rounds_r2[best_idx]})")

# Plot R² scores over rounds
fig, axes = plt.subplots(1, len(target_feature_names), figsize=(7 * len(target_feature_names), 5))
if len(target_feature_names) == 1:
    axes = [axes]

for i, (name, ax) in enumerate(zip(target_feature_names, axes)):
    key = f'r2_feature_{i}'
    r2_history = Output_History.metrics_centralized[key]
    rounds_r2 = [rnd for rnd, _ in r2_history]
    scores = [score for _, score in r2_history]
    
    ax.plot(rounds_r2, scores, marker='.', markersize=3)
    ax.set_title(f'R² for {name}', fontsize=14)
    ax.set_xlabel('Round')
    ax.set_ylabel('R² Score')
    ax.grid(True, alpha=0.3)
    ax.axhline(y=max(scores), color='r', linestyle='--', alpha=0.5, label=f'Best: {max(scores):.4f}')
    ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "r2_scores_plot.png"), dpi=150)
plt.show()